## ⚠️ Important Notebook Execution Guide

This notebook contains the Time-Block Baseline CNN training and evaluation workflow.

### First-time training

Run the notebook in this order:

**RESUME / SETUP → Dataset → TensorFlow Dataset → Model Preparation → Training → Model Loading → Evaluation**

For a new training run, set:

`RETRAIN = True`

Keep:

`RUN_SMOKE_TEST = False`

Do not use the old source-group or session-aware checkpoints for the time-block experiment.

### Evaluation after the model has already been trained

After a Colab runtime restart, run:

**RESUME / SETUP → Time-Block Dataset → TensorFlow Dataset → Load Saved Model → Evaluation**

Do NOT run the training section again when only evaluation is required.

### ⚠️ Do NOT use "Run All" for evaluation.

Running the training section with `RETRAIN = True` will start a new training run.

# Fresh vs Rotten Fruit Classification — Baseline CNN

This notebook develops and trains a baseline Convolutional Neural Network (CNN)
for six-class fresh-versus-rotten fruit classification.

## Objective

Build a baseline CNN that takes a preprocessed fruit image of size
224 × 224 × 3 and predicts one of the six fruit classes.

## Classes

- Fresh Apple
- Fresh Banana
- Fresh Orange
- Rotten Apple
- Rotten Banana
- Rotten Orange

## 1. Time-Block Baseline CNN Development

This notebook trains a Baseline CNN from scratch using the verified
time-block dataset split.

The time-block split uses contiguous source-photo blocks and a minimum
60-second capture-time gap between different dataset splits.

Training will be performed in Google Colab using the available GPU.

This experiment is separate from the earlier source-group and
session-aware Baseline CNN experiments.

In [ ]:
# =========================================================
# RESUME / SETUP
# Run this cell first after every fresh Colab runtime.
# =========================================================

import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------

SEED = 42

RETRAIN = False
RUN_SMOKE_TEST = False

# Reproducibility
import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)

IMAGE_SIZE = (
    224,
    224
)

BATCH_SIZE = 32

CLASS_NAMES = [
    "freshapples",
    "freshbanana",
    "freshoranges",
    "rottenapples",
    "rottenbanana",
    "rottenoranges",
]

NUM_CLASSES = len(CLASS_NAMES)


# ---------------------------------------------------------
# 2. Reproducibility
# ---------------------------------------------------------

np.random.seed(SEED)

tf.keras.utils.set_random_seed(SEED)


# ---------------------------------------------------------
# 3. Repository
# ---------------------------------------------------------

REPO_URL = (
    "https://github.com/"
    "onkar-726/fresh-rotten-fruit-classification.git"
)

REPO_DIR = Path(
    "/content/fresh-rotten-fruit-classification"
)


if not REPO_DIR.exists():

    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO_DIR)
        ],
        check=True
    )

else:

    subprocess.run(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "pull",
            "--ff-only"
        ],
        check=True
    )


os.chdir(REPO_DIR)


if str(REPO_DIR) not in sys.path:

    sys.path.insert(
        0,
        str(REPO_DIR)
    )


print(
    "Repository:",
    REPO_DIR
)

print(
    "Working directory:",
    Path.cwd()
)


# ---------------------------------------------------------
# 4. Install requirements
# ---------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        "requirements.txt",
        "-q"
    ],
    check=True
)


print(
    "Project requirements installed."
)


# ---------------------------------------------------------
# 5. Download / restore Kaggle dataset
# ---------------------------------------------------------

import kagglehub


dataset_path = kagglehub.dataset_download(
    "sriramr/fruits-fresh-and-rotten-for-classification"
)


DATASET_ROOT = (
    Path(dataset_path)
    / "dataset"
)


assert DATASET_ROOT.exists(), (
    f"Dataset root not found: {DATASET_ROOT}"
)


print(
    "Dataset root:",
    DATASET_ROOT
)

# ---------------------------------------------------------
# Google Drive project root
# ---------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/fruit_project"
)

# ---------------------------------------------------------
# 6. Time-block split paths
# ---------------------------------------------------------

TIME_BLOCK_DIR = (
    REPO_DIR
    / "data"
    / "time_block"
)


TIME_BLOCK_FILES = {
    "train": (
        TIME_BLOCK_DIR
        / "train_time_block.csv"
    ),
    "validation": (
        TIME_BLOCK_DIR
        / "validation_time_block.csv"
    ),
    "test": (
        TIME_BLOCK_DIR
        / "test_time_block.csv"
    ),
}


for split_name, file_path in TIME_BLOCK_FILES.items():

    assert file_path.exists(), (
        f"Missing {split_name} CSV: {file_path}"
    )


print(
    "\nTime-block CSV files verified."
)


# ---------------------------------------------------------
# 7. Load the time-block split metadata
# ---------------------------------------------------------

train_df = pd.read_csv(
    TIME_BLOCK_FILES["train"]
)

validation_df = pd.read_csv(
    TIME_BLOCK_FILES["validation"]
)

test_df = pd.read_csv(
    TIME_BLOCK_FILES["test"]
)


print(
    "Training rows  :",
    len(train_df)
)

print(
    "Validation rows:",
    len(validation_df)
)

print(
    "Testing rows   :",
    len(test_df)
)


print(
    "Total rows     :",
    len(train_df)
    + len(validation_df)
    + len(test_df)
)


# ---------------------------------------------------------
# 8. Verify the expected time-block split
# ---------------------------------------------------------

assert len(train_df) == 8280
assert len(validation_df) == 1782
assert len(test_df) == 1935

assert (
    len(train_df)
    + len(validation_df)
    + len(test_df)
    == 11997
)


for split_name, dataframe in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}.items():

    assert set(
        dataframe["class_name"]
    ) == set(CLASS_NAMES), (
        f"Missing class in {split_name}"
    )


print(
    "\nTime-block split verification: PASSED"
)


# ---------------------------------------------------------
# 9. Environment information
# ---------------------------------------------------------

print(
    "\nTensorFlow version:",
    tf.__version__
)

print(
    "Python version:",
    sys.version.split()[0]
)

print(
    "Image size:",
    IMAGE_SIZE
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Number of classes:",
    NUM_CLASSES
)

print(
    "RETRAIN:",
    RETRAIN
)


print(
    "\nRESUME / SETUP completed successfully."
)

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Baseline CNN Architecture

The baseline model is designed as a simple convolutional neural network
trained from scratch.

### Architecture

Input: 224 × 224 × 3

1. Convolutional layer — 32 filters
2. ReLU activation
3. Max pooling
4. Convolutional layer — 64 filters
5. ReLU activation
6. Max pooling
7. Convolutional layer — 128 filters
8. ReLU activation
9. Max pooling
10. Global Average Pooling
11. Dense layer — 128 units
12. Dropout
13. Output layer — 6 units with softmax

The model will serve as the baseline CNN for the six-class fruit
freshness classification task.

## 3. Build the Baseline CNN

The baseline CNN is trained from scratch using the preprocessed
224 × 224 × 3 fruit images.

The network uses three convolutional blocks followed by global
average pooling and fully connected classification layers.

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),

    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(6, activation="softmax")
])

model.summary()

## 4. Compile the Baseline CNN

The CNN is compiled using:

- Sparse categorical cross-entropy as the loss function
- Adam as the optimizer
- Accuracy as the evaluation metric

Integer class labels are used because the dataset labels are encoded as
class indices from 0 to 5.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

print("Model compiled successfully.")

## 5. Baseline CNN Forward-Pass Test

Before training, the model is tested with a single input tensor to verify
that the input shape and six-class output are correct.

In [ ]:
import numpy as np

sample_input = np.random.random((1, 224, 224, 3)).astype(np.float32)

prediction = model.predict(sample_input, verbose=0)

print("Input shape:", sample_input.shape)
print("Output shape:", prediction.shape)
print("Output probabilities:", prediction)
print("Probability sum:", prediction.sum())

## 6. Connect the Time-Block Dataset

The Baseline CNN uses the verified time-block train, validation, and test
CSV files created during preprocessing.

The split metadata is loaded from the project repository, while the actual
images are accessed from the Kaggle dataset in Google Colab.

The time-block test split is kept completely separate from the training
and validation splits.

In [ ]:
# ---------------------------------------------------------
# Connect the Time-Block Dataset
# ---------------------------------------------------------

from pathlib import Path
import pandas as pd


TIME_BLOCK_DIR = (
    REPO_DIR
    / "data"
    / "time_block"
)


train_df = pd.read_csv(
    TIME_BLOCK_DIR
    / "train_time_block.csv"
)

validation_df = pd.read_csv(
    TIME_BLOCK_DIR
    / "validation_time_block.csv"
)

test_df = pd.read_csv(
    TIME_BLOCK_DIR
    / "test_time_block.csv"
)


print(
    "Training rows  :",
    len(train_df)
)

print(
    "Validation rows:",
    len(validation_df)
)

print(
    "Testing rows   :",
    len(test_df)
)

print(
    "Total rows     :",
    len(train_df)
    + len(validation_df)
    + len(test_df)
)


assert len(train_df) == 8280
assert len(validation_df) == 1782
assert len(test_df) == 1935

assert (
    len(train_df)
    + len(validation_df)
    + len(test_df)
    == 11997
)


print(
    "\nTime-block dataset loaded successfully."
)

In [ ]:
# ---------------------------------------------------------
# Verify the time-block split
# ---------------------------------------------------------

for split_name, dataframe in {
    "Training": train_df,
    "Validation": validation_df,
    "Testing": test_df,
}.items():

    print(
        f"\n{split_name}: "
        f"{len(dataframe)} images"
    )

    print(
        dataframe["class_name"]
        .value_counts()
        .sort_index()
    )


# ---------------------------------------------------------
# Dynamic split-size checks
# ---------------------------------------------------------

expected_total_images = (
    len(train_df)
    + len(validation_df)
    + len(test_df)
)

print(
    "\nTotal time-block images:",
    expected_total_images
)

assert expected_total_images == 11997


# ---------------------------------------------------------
# Verify all six classes are present
# ---------------------------------------------------------

for split_name, dataframe in {
    "Training": train_df,
    "Validation": validation_df,
    "Testing": test_df,
}.items():

    assert set(
        dataframe["class_name"]
    ) == set(CLASS_NAMES), (
        f"Missing class in {split_name}"
    )


# ---------------------------------------------------------
# Verify source-group separation
# ---------------------------------------------------------

train_sources = set(
    train_df["source_id"]
)

validation_sources = set(
    validation_df["source_id"]
)

test_sources = set(
    test_df["source_id"]
)


train_validation_overlap = (
    train_sources
    & validation_sources
)

train_test_overlap = (
    train_sources
    & test_sources
)

validation_test_overlap = (
    validation_sources
    & test_sources
)


print(
    "\nTrain ↔ Validation source overlap:",
    len(train_validation_overlap)
)

print(
    "Train ↔ Test source overlap:",
    len(train_test_overlap)
)

print(
    "Validation ↔ Test source overlap:",
    len(validation_test_overlap)
)


assert len(
    train_validation_overlap
) == 0

assert len(
    train_test_overlap
) == 0

assert len(
    validation_test_overlap
) == 0


print(
    "\nTime-block split verification: PASSED"
)

## 7. Create TensorFlow Datasets

In [ ]:
CLASS_NAMES = [
    "freshapples",
    "freshbanana",
    "freshoranges",
    "rottenapples",
    "rottenbanana",
    "rottenoranges",
]

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32


def load_and_preprocess_image(image_path, label):
    image_bytes = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0

    return image, label


def create_dataset(dataframe, shuffle=False):
    image_paths = [
        str(DATASET_ROOT / image_path)
        for image_path in dataframe["image_path"]
    ]

    labels = (
        dataframe["class_name"]
        .map(CLASS_TO_INDEX)
        .to_numpy(dtype=np.int32)
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


train_ds = create_dataset(train_df, shuffle=True)
validation_ds = create_dataset(validation_df, shuffle=False)
test_ds = create_dataset(test_df, shuffle=False)

print("TensorFlow datasets created successfully.")

In [ ]:
for images, labels in train_ds.take(1):
    print("Batch image shape :", images.shape)
    print("Batch label shape :", labels.shape)
    print("Image dtype       :", images.dtype)
    print("Minimum pixel     :", tf.reduce_min(images).numpy())
    print("Maximum pixel     :", tf.reduce_max(images).numpy())

    predictions = model(images, training=False)

    print("CNN output shape  :", predictions.shape)

## 8. CNN Training Smoke Test

Before starting full training, a very small training run is used to verify
that the model, loss function, optimizer, and real training dataset work
together correctly.

In [ ]:
# ---------------------------------------------------------
# CNN Training Smoke Test
# ---------------------------------------------------------

if RUN_SMOKE_TEST:

    print(
        "RUN_SMOKE_TEST=True"
    )

    smoke_train_ds = train_ds.take(2)
    smoke_validation_ds = validation_ds.take(1)

    history_smoke = model.fit(
        smoke_train_ds,
        validation_data=smoke_validation_ds,
        epochs=1,
        verbose=1
    )

    print(
        "\nSmoke test completed successfully."
    )

else:

    print(
        "RUN_SMOKE_TEST=False"
    )

    print(
        "Smoke test skipped."
    )

## 9. Prepare Time-Block Baseline CNN Training

The Baseline CNN is trained from scratch using the time-block training split.

The best model is saved with a unique experiment-specific filename so it
cannot be confused with the earlier source-group or session-aware models.

Training artifacts are stored in Google Drive so they survive Colab runtime
resets.

Set `RETRAIN = True` in the RESUME / SETUP cell only when a new training run
is required.

In [ ]:
# ---------------------------------------------------------
# Prepare Time-Block Baseline CNN training
# ---------------------------------------------------------

import tensorflow as tf
from pathlib import Path


# ---------------------------------------------------------
# Google Drive
# ---------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive"
)


DRIVE_ROOT = Path(
    "/content/drive/MyDrive/fruit_project"
)

MODEL_DIR = (
    DRIVE_ROOT
    / "models"
)

HISTORY_DIR = (
    DRIVE_ROOT
    / "results"
    / "baseline_cnn_time_block"
    / "history"
)

RESULTS_DIR = (
    DRIVE_ROOT
    / "results"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HISTORY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# Unique experiment names
# ---------------------------------------------------------

MODEL_NAME = (
    "baseline_cnn_time_block_best.keras"
)

MODEL_PATH = (
    MODEL_DIR
    / MODEL_NAME
)

HISTORY_CSV_PATH = (
    HISTORY_DIR
    / "baseline_cnn_time_block_history.csv"
)

HISTORY_JSON_PATH = (
    HISTORY_DIR
    / "baseline_cnn_time_block_history.json"
)

ACCURACY_PLOT_PATH = (
    HISTORY_DIR
    / "baseline_cnn_time_block_accuracy.png"
)

LOSS_PLOT_PATH = (
    HISTORY_DIR
    / "baseline_cnn_time_block_loss.png"
)


# ---------------------------------------------------------
# Start with a clean TensorFlow session
# ---------------------------------------------------------

tf.keras.backend.clear_session()


# ---------------------------------------------------------
# Create a fresh Baseline CNN
# ---------------------------------------------------------

model = tf.keras.Sequential([

    tf.keras.Input(
        shape=IMAGE_SIZE + (3,)
    ),

    tf.keras.layers.Conv2D(
        32,
        (3, 3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(
        (2, 2)
    ),

    tf.keras.layers.Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(
        (2, 2)
    ),

    tf.keras.layers.Conv2D(
        128,
        (3, 3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D(
        (2, 2)
    ),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dropout(
        0.5
    ),

    tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])


# ---------------------------------------------------------
# Compile
# ---------------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)


# ---------------------------------------------------------
# Training configuration
# ---------------------------------------------------------

EPOCHS = 15


callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(HISTORY_CSV_PATH),
        append=False,
    ),
]


print(
    "Time-block Baseline CNN prepared."
)

print(
    "Model path:",
    MODEL_PATH
)

print(
    "History CSV:",
    HISTORY_CSV_PATH
)

print(
    "Accuracy plot:",
    ACCURACY_PLOT_PATH
)

print(
    "Loss plot:",
    LOSS_PLOT_PATH
)

print(
    "Epoch limit:",
    EPOCHS
)

print(
    "RETRAIN:",
    RETRAIN
)

## 10. Time-Block Baseline CNN Training — Results

This section trains the Baseline CNN using the verified time-block training
split and uses the time-block validation split for monitoring.

The final training results will be recorded here only after the new
time-block training run is completed.

The following artifacts are saved to Google Drive:

- Best model checkpoint
- Training history CSV
- Training history JSON
- Accuracy plot
- Loss plot

The time-block test-set results are reported separately in Section 12
after the trained model is loaded.

In [ ]:
# ---------------------------------------------------------
# Time-Block Baseline CNN Training
# ---------------------------------------------------------

import os
import json
import pandas as pd
import matplotlib.pyplot as plt


if RETRAIN:

    print("RETRAIN=True")
    print("Starting time-block Baseline CNN training...\n")

    history = model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    # -----------------------------------------------------
    # Prepare history directory
    # -----------------------------------------------------

    os.makedirs(HISTORY_DIR, exist_ok=True)

    # -----------------------------------------------------
    # Save complete history - CSV
    # -----------------------------------------------------

    pd.DataFrame(history.history).to_csv(
        HISTORY_CSV_PATH,
        index=False
    )

    # -----------------------------------------------------
    # Save complete history - JSON
    # -----------------------------------------------------

    with open(HISTORY_JSON_PATH, "w") as f:
        json.dump(history.history, f, indent=2)

    # -----------------------------------------------------
    # Accuracy plot
    # -----------------------------------------------------

    plt.figure(figsize=(8, 5))

    plt.plot(
        history.history["accuracy"],
        label="Training Accuracy"
    )

    plt.plot(
        history.history["val_accuracy"],
        label="Validation Accuracy"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Time-Block Baseline CNN Accuracy")
    plt.legend()
    plt.grid(True)

    plt.savefig(
        ACCURACY_PLOT_PATH,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    # -----------------------------------------------------
    # Loss plot
    # -----------------------------------------------------

    plt.figure(figsize=(8, 5))

    plt.plot(
        history.history["loss"],
        label="Training Loss"
    )

    plt.plot(
        history.history["val_loss"],
        label="Validation Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Time-Block Baseline CNN Loss")
    plt.legend()
    plt.grid(True)

    plt.savefig(
        LOSS_PLOT_PATH,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    # -----------------------------------------------------
    # Confirm saved artifacts
    # -----------------------------------------------------

    print("\nTraining history saved:")

    print(HISTORY_CSV_PATH)
    print(HISTORY_JSON_PATH)
    print(ACCURACY_PLOT_PATH)
    print(LOSS_PLOT_PATH)


else:

    print("RETRAIN=False")
    print("Training skipped.")
    print("Use the model-loading section to load the saved time-block model.")

## 11. Load the Saved Time-Block Baseline CNN

This section loads the Baseline CNN trained specifically on the time-block
training split.

The expected checkpoint is:

`baseline_cnn_time_block_best.keras`

The model can be restored in either of two ways:

1. Google Drive — recommended for repeated use after Colab runtime resets.
2. File upload — useful when the `.keras` file is stored locally.

**Important:** Do not load a checkpoint from the source-group or
session-aware experiments for time-block evaluation.

In [ ]:
# ---------------------------------------------------------
# Load the time-block Baseline CNN
# ---------------------------------------------------------

from pathlib import Path
import tensorflow as tf


EXPECTED_MODEL_NAME = (
    "baseline_cnn_time_block_best.keras"
)


# ---------------------------------------------------------
# Choose how to restore the model
#
# "drive"   -> load from Google Drive
# "upload"  -> upload the .keras file manually
# ---------------------------------------------------------

LOAD_METHOD = "drive"


# ---------------------------------------------------------
# Option A — Google Drive
# ---------------------------------------------------------

if LOAD_METHOD == "drive":

    from google.colab import drive

    drive.mount(
        "/content/drive"
    )


    MODEL_PATH = Path(
        "/content/drive/MyDrive/"
        "fruit_project/models/"
        "baseline_cnn_time_block_best.keras"
    )

    assert MODEL_PATH.exists(), (
        f"Expected model not found:\n{MODEL_PATH}\n\n"
        "Train the model with RETRAIN=True first, "
        "or change LOAD_METHOD to 'upload'."
    )


# ---------------------------------------------------------
# Option B — Upload from your computer
# ---------------------------------------------------------

elif LOAD_METHOD == "upload":

    from google.colab import files

    uploaded = files.upload()


    assert len(uploaded) > 0, (
        "No model file was uploaded."
    )


    uploaded_names = list(
        uploaded.keys()
    )


    assert len(uploaded_names) == 1, (
        "Upload exactly one .keras model file."
    )


    uploaded_name = uploaded_names[0]


    assert uploaded_name == EXPECTED_MODEL_NAME, (
        f"Wrong model file uploaded: {uploaded_name}\n"
        f"Expected: {EXPECTED_MODEL_NAME}"
    )


    MODEL_PATH = Path(
        "/content"
    ) / uploaded_name


# ---------------------------------------------------------
# Reject an invalid loading method
# ---------------------------------------------------------

else:

    raise ValueError(
        "LOAD_METHOD must be 'drive' or 'upload'."
    )


# ---------------------------------------------------------
# Final filename safety check
# ---------------------------------------------------------

assert (
    MODEL_PATH.name
    == EXPECTED_MODEL_NAME
), (
    f"Wrong model filename: {MODEL_PATH.name}"
)


print(
    "Loading model:",
    MODEL_PATH
)


# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------

model = tf.keras.models.load_model(
    MODEL_PATH
)


print(
    "\nTime-block Baseline CNN loaded successfully."
)

print(
    "Model filename:",
    MODEL_PATH.name
)

print(
    "Number of parameters:",
    model.count_params()
)

## 🔄 Notebook Execution Guide After Colab Restart

This notebook contains the Time-Block Baseline CNN training and evaluation workflow.

### First-time training

For the new Time-Block Baseline CNN experiment:

1. Run the **RESUME / SETUP** cell.
2. Run the dataset and input-pipeline sections.
3. Keep:
   `RETRAIN = True`
   and
   `RUN_SMOKE_TEST = False`
4. Run the model preparation section.
5. Run the official Time-Block Baseline CNN training section.

The best model is saved as:

`baseline_cnn_time_block_best.keras`

Training history and plots are saved to Google Drive.

### Evaluation after the model has been trained

After a Colab runtime restart:

1. Run the **RESUME / SETUP** cell.
2. Recreate the TensorFlow datasets.
3. Run the **Time-Block Baseline CNN model-loading section**.
4. Run the test-set evaluation sections.

The saved model filename is:

`baseline_cnn_time_block_best.keras`

### Important

The required checkpoint for this experiment is:

`baseline_cnn_time_block_best.keras`

Do NOT use checkpoints from the earlier source-group or session-aware experiments for Time-Block evaluation.

Do NOT run the training section again when only evaluation is required.

### Evaluation workflow after restart

`RESUME / SETUP → Dataset/Input Pipeline → Model Loading → Test Evaluation`

This workflow allows the saved Time-Block Baseline CNN to be evaluated without retraining.

## 12. Time-Block Baseline CNN Test Set Evaluation

The saved best Time-Block Baseline CNN is evaluated on the verified
time-block test split.

This section performs evaluation only. It does not train the model.

Before running this section after a Colab runtime restart:

- run the RESUME / SETUP cell;
- recreate `test_ds`;
- load `baseline_cnn_time_block_best.keras`.

The time-block test split contains 1,935 images from 215 source photos.

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

## 13. Time-Block Baseline CNN Precision, Recall and F1-score

The trained Time-Block Baseline CNN is evaluated using the complete
time-block test set.

Precision, recall and F1-score are calculated for each fruit class using
the model predictions and true class labels.

These metrics provide more detailed information than accuracy alone.

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Get prediction probabilities for the complete test set.
y_prob = model.predict(test_ds, verbose=1)

# Get the true class labels from the test set.
y_true = np.concatenate(
    [labels.numpy() for _, labels in test_ds]
)

# Convert probabilities into predicted class indices.
y_pred = np.argmax(y_prob, axis=1)

print("Number of true labels:", len(y_true))
print("Number of predictions:", len(y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)

## 14. Time-Block Baseline CNN Confusion Matrix

The confusion matrix shows how the Time-Block Baseline CNN classified the
six fruit classes on the time-block test set.

Rows represent the true classes and columns represent the predicted classes.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_true, y_pred)

print("Confusion matrix:")
print(cm)

plt.figure(figsize=(8, 6))

plt.imshow(cm, interpolation="nearest")
plt.title("Baseline CNN Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(
    np.arange(len(CLASS_NAMES)),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    np.arange(len(CLASS_NAMES)),
    CLASS_NAMES
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

## 15. Save Baseline CNN Evaluation Results

The final Baseline CNN evaluation results are preserved separately from
the notebook output for easier use in the project report and presentation.

In [ ]:
# ---------------------------------------------------------
# Save Time-Block Baseline CNN evaluation results
# ---------------------------------------------------------

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)


# ---------------------------------------------------------
# Persistent Google Drive results directory
# ---------------------------------------------------------

DRIVE_RESULTS_DIR = (
    DRIVE_ROOT
    / "results"
    / "baseline_cnn_time_block"
)

DRIVE_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# Classification report
# ---------------------------------------------------------

report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)


report_path = (
    DRIVE_RESULTS_DIR
    / "baseline_cnn_time_block_classification_report.txt"
)


report_with_summary = (
    "Time-Block Baseline CNN Evaluation\n"
    "===================================\n\n"
    f"Test images: {len(test_df)}\n"
    f"Test accuracy: {test_accuracy:.6f}\n"
    f"Test loss: {test_loss:.6f}\n\n"
    + report
)


report_path.write_text(
    report_with_summary,
    encoding="utf-8"
)


print(
    "Classification report saved to:",
    report_path
)


# ---------------------------------------------------------
# Confusion matrix
# ---------------------------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=np.arange(NUM_CLASSES)
)


print(
    "\nConfusion matrix:"
)

print(
    cm
)


plt.figure(
    figsize=(8, 6)
)

plt.imshow(
    cm,
    interpolation="nearest"
)

plt.title(
    "Time-Block Baseline CNN Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "True Class"
)


plt.xticks(
    np.arange(NUM_CLASSES),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    np.arange(NUM_CLASSES),
    CLASS_NAMES
)


for i in range(
    NUM_CLASSES
):

    for j in range(
        NUM_CLASSES
    ):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )


plt.tight_layout()


cm_path = (
    DRIVE_RESULTS_DIR
    / "baseline_cnn_time_block_confusion_matrix.png"
)


plt.savefig(
    cm_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print(
    "Confusion matrix saved to:",
    cm_path
)


# ---------------------------------------------------------
# Verify result files
# ---------------------------------------------------------

assert report_path.exists()
assert cm_path.exists()


print(
    "\nTime-block Baseline CNN evaluation artifacts "
    "saved successfully."
)

## 16. Time-Block Baseline CNN Original-Only Test Evaluation

The time-block test split contains augmented copies of the source images.

This section creates an original-only test view containing one non-augmented
image per source group.

The trained Time-Block Baseline CNN is evaluated on this original-only view
without any retraining.

The original-only results are reported separately from the full test-set
results.

In [ ]:
# ---------------------------------------------------------
# Create original-only view of the time-block test split
# ---------------------------------------------------------

import re
import numpy as np
import pandas as pd


AUGMENTATION_PATTERNS = [
    r"^rotated_by_\d+_",
    r"^vertical_flip_",
    r"^translation_",
    r"^saltandpepper_",
]


def is_original_image(image_path):
    filename = Path(image_path).name

    return not any(
        re.match(
            pattern,
            filename
        )
        for pattern in AUGMENTATION_PATTERNS
    )


# Keep only non-augmented images.
original_test_df = (
    test_df[
        test_df["image_path"].apply(
            is_original_image
        )
    ]
    .copy()
)


# ---------------------------------------------------------
# Verify one original image per source group
# ---------------------------------------------------------

print(
    "Original-only test images:",
    len(original_test_df)
)

print(
    "Unique source IDs:",
    original_test_df["source_id"].nunique()
)

print(
    "Unique source filenames:",
    original_test_df["source_filename"].nunique()
)


assert (
    len(original_test_df)
    == original_test_df["source_id"].nunique()
)


assert all(
    original_test_df["image_path"].apply(
        is_original_image
    )
)


assert set(
    original_test_df["class_name"]
) == set(CLASS_NAMES)


print(
    "\nOriginal-only class distribution:"
)

print(
    original_test_df["class_name"]
    .value_counts()
    .sort_index()
)


print(
    "\nOriginal-only test set verification: PASSED"
)

## 17. Time-Block Baseline CNN Original-Only Metrics

The saved Time-Block Baseline CNN is evaluated on the original-only test
view.

No training or fine-tuning is performed in this section.

In [ ]:
# ---------------------------------------------------------
# Build original-only TensorFlow dataset
# ---------------------------------------------------------

original_test_ds = create_dataset(
    original_test_df,
    shuffle=False
)


# ---------------------------------------------------------
# Evaluate model
# ---------------------------------------------------------

original_test_loss, original_test_accuracy = (
    model.evaluate(
        original_test_ds,
        verbose=1
    )
)


print(
    "\nTime-Block Baseline CNN Original-Only Results"
)

print(
    "Test loss:",
    original_test_loss
)

print(
    "Test accuracy:",
    original_test_accuracy
)


# ---------------------------------------------------------
# Generate predictions
# ---------------------------------------------------------

original_y_prob = model.predict(
    original_test_ds,
    verbose=1
)


original_y_pred = np.argmax(
    original_y_prob,
    axis=1
)


original_y_true = np.concatenate(
    [
        labels.numpy()
        for _, labels in original_test_ds
    ]
)


print(
    "Number of true labels:",
    len(original_y_true)
)

print(
    "Number of predictions:",
    len(original_y_pred)
)

## 18. Time-Block Baseline CNN Original-Only Classification Report

In [ ]:
from sklearn.metrics import classification_report


original_report = classification_report(
    original_y_true,
    original_y_pred,
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)


print(
    "Time-Block Baseline CNN Original-Only "
    "Classification Report:\n"
)

print(
    original_report
)

## 19. Save Original-Only Baseline CNN Results

In [ ]:
# ---------------------------------------------------------
# Save original-only Baseline CNN results
# ---------------------------------------------------------

ORIGINAL_RESULTS_DIR = (
    DRIVE_ROOT
    / "results"
    / "baseline_cnn_time_block"
)

ORIGINAL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# Save classification report
# ---------------------------------------------------------

original_report_path = (
    ORIGINAL_RESULTS_DIR
    / "baseline_cnn_time_block_original_only_classification_report.txt"
)


original_report_text = (
    "Time-Block Baseline CNN Original-Only Evaluation\n"
    "=================================================\n\n"
    f"Original-only test images: {len(original_test_df)}\n"
    f"Test loss: {original_test_loss:.6f}\n"
    f"Test accuracy: {original_test_accuracy:.6f}\n\n"
    + original_report
)


original_report_path.write_text(
    original_report_text,
    encoding="utf-8"
)


print(
    "Original-only classification report saved to:",
    original_report_path
)


# ---------------------------------------------------------
# Save original-only confusion matrix
# ---------------------------------------------------------

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt


original_cm = confusion_matrix(
    original_y_true,
    original_y_pred,
    labels=np.arange(NUM_CLASSES)
)


print(
    "\nOriginal-only confusion matrix:"
)

print(
    original_cm
)


plt.figure(
    figsize=(8, 6)
)

plt.imshow(
    original_cm,
    interpolation="nearest"
)

plt.title(
    "Time-Block Baseline CNN Original-Only Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "True Class"
)


plt.xticks(
    np.arange(NUM_CLASSES),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    np.arange(NUM_CLASSES),
    CLASS_NAMES
)


for i in range(NUM_CLASSES):

    for j in range(NUM_CLASSES):

        plt.text(
            j,
            i,
            original_cm[i, j],
            ha="center",
            va="center"
        )


plt.tight_layout()


original_cm_path = (
    ORIGINAL_RESULTS_DIR
    / "baseline_cnn_time_block_original_only_confusion_matrix.png"
)


plt.savefig(
    original_cm_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print(
    "Original-only confusion matrix saved to:",
    original_cm_path
)


# ---------------------------------------------------------
# Final verification
# ---------------------------------------------------------

assert original_report_path.exists()
assert original_cm_path.exists()


print(
    "\nOriginal-only Baseline CNN result export "
    "verification passed."
)

## Time-Block Baseline CNN — Current Status

The notebook is prepared for the Time-Block Baseline CNN experiment.

Completed preparation:

- Time-block dataset connected
- Time-block split size verified
- Baseline CNN architecture defined
- Training configuration defined
- Google Drive model checkpoint path defined
- Training history CSV/JSON paths defined
- Accuracy and loss plot paths defined
- Resume workflow defined
- Original-only test evaluation workflow prepared

Pending:

- Train the Baseline CNN on the time-block training split
- Evaluate the trained model on the time-block test split
- Evaluate the original-only time-block test view
- Export the final time-block results

Do not report final performance values until the new time-block training and evaluation have been completed.